### Topic modeling

In [1]:
!pip install bertopic
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.3 MB/s eta 0:00:00


In [2]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import re
from collections import Counter

In [3]:
df_pelis = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/peliculas_limpio.csv")

In [4]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/usuarios.csv")

## Preprocesado

In [5]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

In [6]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + ". "
    # + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

## Entrenamiento

In [7]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
embeddings = model.encode(df_pelis["texto"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

In [9]:
# Instanciación con configuración mínima
topic_model = BERTopic(
    embedding_model=model,
    calculate_probabilities=True,
    verbose=True
    )

# Entrenamiento (pasando embeddings pre-computados)
topics, probs = topic_model.fit_transform(df_pelis["texto"].tolist(), embeddings)

2026-06-14 22:49:29,202 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-14 22:50:08,410 - BERTopic - Dimensionality - Completed ✓
2026-06-14 22:50:08,412 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-14 22:50:09,696 - BERTopic - Cluster - Completed ✓
2026-06-14 22:50:09,712 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-14 22:50:10,031 - BERTopic - Representation - Completed ✓


In [10]:
# Número de tópicos encontrados (excluye -1 = outliers)
n_topics = len(topic_model.get_topics()) - 1
print(f"Tópicos encontrados: {n_topics}")
print(f"Outliers (tópico -1): {topics.count(-1)}")

Tópicos encontrados: 65
Outliers (tópico -1): 2353


In [11]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2353,-1_de_la_en_un,"[de, la, en, un, una, su, que, el, drama, come...",[Ella es el chico. Cuando su hermano decide de...
1,0,567,0_policía_crimen_agente_acción,"[policía, crimen, agente, acción, de, detectiv...",[Canción triste de Hill Street. La vida y el t...
2,1,119,1_familia_padre_su_madre,"[familia, padre, su, madre, una, hijo, drama, ...",[Los chicos están bien. Dos niños concebidos p...
3,2,117,2_alienígena_ficción_ciencia_alienígenas,"[alienígena, ficción, ciencia, alienígenas, ti...",[Planet 51. Una civilización alienígena es inv...
4,3,113,3_china_marciales_artes_acción,"[china, marciales, artes, acción, japón, la, d...",[El reino prohibido. Un descubrimiento de un a...
5,4,94,4_aventura_fantasía_rey_el,"[aventura, fantasía, rey, el, animación, dragó...","[Beowulf, la leyenda. En una tierra sitiada, B..."
6,5,90,5_romance_matrimonio_boda_mujer,"[romance, matrimonio, boda, mujer, hombre, sol...",[American Pie: Menuda boda!. Los años han pasa...
7,6,85,6_escuela_teacher_instituto_estudiante,"[escuela, teacher, instituto, estudiante, estu...",[Escuela de rock. Tras ser expulsado de su ban...
8,7,74,7_música_rock_jazz_banda,"[música, rock, jazz, banda, cantante, pianista...",[Purple Rain. Una historia humana de supervive...
9,8,68,8_prisión_crimen_rehén_banco,"[prisión, crimen, rehén, banco, fuga, un, robo...",[Dos fugitivos. Un atracador estúpido toma a J...


In [12]:
# Documentos más representativos del tópico 1
print("Documentos representativos del tópico 1:\n")
for doc in topic_model.get_representative_docs(1):
    print(doc[:100])
    print("---")

Documentos representativos del tópico 1:

Los chicos están bien. Dos niños concebidos por inseminación artificial llevan a su padre biológico 
---
El orfanato. Una mujer regresa con su familia al hogar de su infancia, que solía ser un orfanato par
---
El silencio. Tras la muerte de su padre, una adolescente sorda queda huérfana, por lo que se va a vi
---


Para cada usuario: concatenar query + histórico y obtener distribución

In [13]:
def build_user_text(usuario_row, pelis_df):
    """Construye el texto concatenado para un usuario"""
    user_text = usuario_row['query']
    
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        nombre = usuario_row[col]
        peli = pelis_df[pelis_df['name'] == nombre]
        if not peli.empty:
            user_text += " " + peli["texto"].values[0]
    
    return user_text

In [14]:
# Agregar columna de texto a usuarios
usuarios['texto'] = usuarios.apply(lambda row: build_user_text(row, df_pelis), axis=1)

In [15]:
_, users_probs = topic_model.transform(usuarios['texto'].tolist())

print(f"users_probs shape: {users_probs.shape}")  # (14, n_topics)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-14 22:50:14,622 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-06-14 22:50:37,795 - BERTopic - Dimensionality - Completed ✓
2026-06-14 22:50:37,797 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-06-14 22:50:37,800 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-06-14 22:50:37,811 - BERTopic - Probabilities - Completed ✓
2026-06-14 22:50:37,812 - BERTopic - Cluster - Completed ✓


users_probs shape: (14, 65)


## Recomendaciones

In [17]:
# Calcular similitud coseno
scores = cosine_similarity(users_probs, probs)

# Crear máscara de películas ya vistas
scores_filtered = scores.copy()

for i, row in usuarios.iterrows():
    historial_names = [row['pelicula_1'], row['pelicula_2'], row['pelicula_3'], 
                       row['pelicula_4'], row['pelicula_5']]
    
    # Encontrar índices de películas en el historial
    historial_indices = df_pelis[df_pelis['name'].isin(historial_names)].index
    
    # Asignar -inf para que no salgan en top-5
    scores_filtered[i, historial_indices] = -np.inf

# Top-5 por usuario
top5_indices = scores.argsort(axis=1)[:, -5:][:, ::-1]

# Ver resultados
for i, row in usuarios.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    for idx in top5_indices[i]:
        pelicula = df_pelis.iloc[idx]
        print(f"  {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {scores[i, idx]:.4f}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
  Un entrenador genial (2022) — 0.9447
  Invencible (2007) — 0.9343
  A por todas (2000) — 0.9316
  Jerry Maguire (1997) — 0.9108
  La leyenda de Bagger Vance (2001) — 0.8782

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
  Causa justa (1995) — 0.9984
  Grita libertad (1988) — 0.9981
  El color púrpura (1986) — 0.9810
  Tiempo de matar (1996) — 0.9803
  Adiós Bafana (2007) — 0.9803

Camila (definido)
Query: Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
  Océanos de fuego (Hidalgo) (2004) — 1.0000
  Dreamer: Camino hacia la victoria (2005) — 1.0000
  Babe: El cerdito en la ciudad (1999) — 1.0000
  Flicka (2006) — 1.0000
  El hombre que susurraba a los caballos (1998) — 1.0000

Tomás (definido)
Query: Algo que haga pensar sobre qué es real y qué es una construcción, con acció

Horrible. Al último le recomienda todo Harry Potter jaj

In [18]:
etiquetas_a_ojo_def = [
    ["suspense", "terror", "drama"],
    ["crimen", "biografía", "historia"],
    ["comedia", "romance", "drama"],
    ["acción", "ciencia ficción", "suspense"],
    ["animación", "drama", "aventura"],
    ["crimen", "acción", "comedia"],
    ["música", "drama", "comedia"],
    ["acción", "crimen", "aventura"],
    ["drama", "romance", "comedia"],
    # el U10 es ambiguo, la etiqueta debe estar mal
]

In [19]:
resultados_eval = []

for user_idx, row in usuarios.head(9).iterrows():
    print(f"\n{'='*70}")
    print(f"{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    generos_esperados = set(etiquetas_a_ojo_def[user_idx])
    print(f"Géneros Esperados: {', '.join(generos_esperados)}")
    print(f"{'='*70}")
    
    generos_recomendados = Counter()
    peliculas_buenas = 0  # con al menos 1 género esperado
    peliculas_malas = []  # sin ningún género esperado
    
    print("\nTop-5 Recomendaciones:")
    for rank, idx in enumerate(top5_indices[user_idx], 1):
        pelicula = df_pelis.iloc[idx]
        score = scores[user_idx, idx]
        
        # Extraer géneros
        generos_str = pelicula['genre'].strip('[]')
        generos_list = [g.strip() for g in generos_str.split(',')]
        generos_pelicula = set(generos_list)
        generos_recomendados.update(generos_list)
        
        # ¿Tiene algún género esperado?
        es_buena = bool(generos_pelicula & generos_esperados)
        if es_buena:
            peliculas_buenas += 1
            marker = "✓"
        else:
            peliculas_malas.append(pelicula['name'])
            marker = "✗"
        
        print(f"  {rank}. [{marker}] {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {score:.4f}")
        print(f"     Géneros: {', '.join(generos_list)}")
    
    # Conteo de géneros
    print(f"\nConteo de Géneros en Recomendaciones:")
    for genero, freq in generos_recomendados.most_common():
        print(f"  {genero}: {freq}")
    
    # Métricas
    generos_capturados = set(generos_recomendados.keys())
    recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
    precision = peliculas_buenas / 5  # top-5
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\nMÉTRICAS:")
    print(f"  Recall (géneros):     {recall:.1%}  ({len(generos_capturados & generos_esperados)}/{len(generos_esperados)})")
    print(f"  Precision (películas): {precision:.1%}  ({peliculas_buenas}/5)")
    print(f"  F1-Score:            {f1:.1%}")
    
    if peliculas_malas:
        print(f"\nPelículas problemáticas (sin géneros esperados):")
        for pelicula in peliculas_malas:
            print(f"    - {pelicula}")
    
    resultados_eval.append({
        'Usuario': row['nombre'],
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'Películas Malas': len(peliculas_malas)
    })

# Resumen
print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN")
print(f"{'='*70}")
df_eval = pd.DataFrame(resultados_eval)
df_eval.to_csv("evaluacion_bertopic.csv", index=False)
print(df_eval.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval['Recall'].mean():.1%}")
print(f"  Precision: {df_eval['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval['F1'].mean():.1%}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: suspense, drama, terror

Top-5 Recomendaciones:
  1. [✗] Un entrenador genial (2022) — 0.9447
     Géneros: comedia, familiar, romance
  2. [✓] Invencible (2007) — 0.9343
     Géneros: biografía, drama, deporte
  3. [✗] A por todas (2000) — 0.9316
     Géneros: comedia, romance, deporte
  4. [✓] Jerry Maguire (1997) — 0.9108
     Géneros: comedia, drama, romance
  5. [✓] La leyenda de Bagger Vance (2001) — 0.8782
     Géneros: drama, fantasía, deporte

Conteo de Géneros en Recomendaciones:
  comedia: 3
  romance: 3
  drama: 3
  deporte: 3
  familiar: 1
  biografía: 1
  fantasía: 1

MÉTRICAS:
  Recall (géneros):     33.3%  (1/3)
  Precision (películas): 60.0%  (3/5)
  F1-Score:            42.9%

Películas problemáticas (sin géneros esperados):
    - Un entrenador genial
    - A por todas

Rodrigo (definido)
Query: Busco algo basado en hechos rea